# AI LIKENESS FEEDBACK

## Roadmap

1. Training data

    1.1 Human writing

    1.2 LLM-generated writing

    1.3 RAID

    1.4 HC3

2.  AI-vs-Human classifier
   (DeBERTa-v3 / RoBERTa)

3. AI-likelihood score

4. sentence/token scores

5. Browser editor

6. red ─ yellow ─ green heatmap

We would make a sentence-level classifier

Sentence 1  🔴
Sentence 2  🟢
Sentence 3  🟡

For each participant, we have -
control_1
control_2

6 AI generations

And 
treatment_1
treatment_2
treatment_3
treatment_4

Setence segmentation of document

Train/validation/test split for participants (57-24) already done in A3.

## Import Libraries and Data

In [22]:
import os
import json
import pandas as pd

In [23]:
path = r"D:\Academics Semester 5\DS501 - Machine Learning\Project\PersonalizationOfLLMs"

In [24]:
with open(path + r"\train_files.txt", "r", encoding = "utf-8") as f:
    train_files = [line.strip() for line in f if line.strip()]

with open(path + r"\test_files.txt", "r", encoding = "utf-8") as f:
    test_files = [line.strip() for line in f if line.strip()]

In [25]:
print("Training participants:", len(train_files))
print("Test participants:", len(test_files))

print(train_files[:2])
print(test_files[:2])


Training participants: 57
Test participants: 24
['d:\\Academics Semester 5\\DS501 - Machine Learning\\Project\\PersonalizationOfLLMs\\personal_style_postedit\\logs\\b417dba7-2b67-4496-b985-fc67c15aa748.json', 'd:\\Academics Semester 5\\DS501 - Machine Learning\\Project\\PersonalizationOfLLMs\\personal_style_postedit\\logs\\32a9cad7-5b1c-480f-832c-1cedb782d226.json']
['d:\\Academics Semester 5\\DS501 - Machine Learning\\Project\\PersonalizationOfLLMs\\personal_style_postedit\\logs\\fe48cc42-b6f0-41a3-b2a0-0c3a1d733997.json', 'd:\\Academics Semester 5\\DS501 - Machine Learning\\Project\\PersonalizationOfLLMs\\personal_style_postedit\\logs\\57f6eb08-1334-4d1d-b32e-7ead2fae69b5.json']


## Load all texts into records

participant_id

│

├── control

│   ├── (AI draft, post-edited text)

│   ├── (AI draft, post-edited text)

│   ├── (AI draft, post-edited text)

│   └── (AI draft, post-edited text)

│

└── pure

    ├── (AI draft, purely human text)
    
    └── (AI draft, purely human text)

In [26]:
records = dict()

for file in train_files:
    with open(file, "r", encoding = "utf-8") as f:
        data = json.load(f)
    
    part_id = data["user_info"]["id"]
    records[part_id] = {"control": [], "pure": []}
    
    for task_id, response in data["responses"].items():
        if not response.get("scenario"):
            continue
        
        if response["model_generation_shown"] == 0:
            records[part_id]["pure"].append((response["model_generation"], response["final_version"]))
        
        elif response["model_generation_shown"] == 1:
            records[part_id]["control"].append((response["model_generation"], response["final_version"])) 

print(len(records))


57


In [27]:
first_id = next(iter(records))

print(first_id)
print(records[first_id]['control'][0][0])
print("-"*5)
print(records[first_id]['control'][0][1])
print("-"*50)
print(records[first_id]['control'][1][0])
print("-"*5)
print(records[first_id]['control'][1][1])
print("-"*50)
print(records[first_id]['control'][2][0])
print("-"*5)
print(records[first_id]['control'][2][1])
print("-"*50)
print(records[first_id]['control'][3][0])
print("-"*5)
print(records[first_id]['control'][3][1])
print("-"*50)
print(records[first_id]['pure'][0][0])
print("-"*5)
print(records[first_id]['pure'][0][1])
print("-"*50)
print(records[first_id]['pure'][1][0])
print("-"*5)
print(records[first_id]['pure'][1][1])

b417dba7-2b67-4496-b985-fc67c15aa748
Hey, I know you’ve been feeling anxious about your career change and uncertain about your marriage. It’s completely normal to feel a bit lost when everything seems to be shifting at once. You’re stepping into new territory, and with that comes a mix of excitement and fear. Remember how resilient you are—you’ve navigated challenges before and grown stronger every time. It’s okay to slow down, take a breath, and give yourself permission to feel whatever comes up. You don’t have to have all the answers right now. Trust that clarity will arrive when the time is right. Lean on the people who care about you and let them remind you of your strengths. You’re not alone in this journey, and every step forward—no matter how small—is progress. I believe in you and your ability to create a future that feels authentic and fulfilling. You’ve got this.
-----
Hey, 
I know you’ve been feeling anxious about your career change and uncertain about the changes ahead - th

## Ai

In [28]:
import nltk
from nltk.tokenize import sent_tokenize

In [31]:
nltk.download("punkt_tab")
sentence_examples = []

for participant_id, participant_data in records.items():
    
    for condition in ("control", "pure"):
        for ai_draft, final_text in participant_data[condition]:
            for sentence in sent_tokenize(ai_draft):
                sentence = sentence.strip()
                if sentence:
                    sentence_examples.append(
                        {
                            "participant_id": participant_id,
                            "text": sentence,
                            "label": 1, # 
                            "source": f"{condition}_ai_draft"
                        }
                    )
    
    for ai_draft, human_text in participant_data["pure"]:
        for sentence in sent_tokenize(human_text):
            sentence = sentence.strip()
            if sentence:
                sentence_examples.append({
                    "participant_id": participant_id,
                    "text": sentence,
                    "label": 0,  # Human
                    "source": "pure_final"
                })

sentence_df = pd.DataFrame(sentence_examples)
print("Sentence examples", len(sentence_df))
print("Participants: ", sentence_df["participant_id"].nunique())
print(sentence_df["label"].value_counts())
sentence_df.head()

Sentence examples 4441
Participants:  57
label
1    3339
0    1102
Name: count, dtype: int64


[nltk_data] Downloading package punkt_tab to C:\Users\Samarth
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,participant_id,text,label,source
0,b417dba7-2b67-4496-b985-fc67c15aa748,"Hey, I know you’ve been feeling anxious about ...",1,control_ai_draft
1,b417dba7-2b67-4496-b985-fc67c15aa748,It’s completely normal to feel a bit lost when...,1,control_ai_draft
2,b417dba7-2b67-4496-b985-fc67c15aa748,"You’re stepping into new territory, and with t...",1,control_ai_draft
3,b417dba7-2b67-4496-b985-fc67c15aa748,Remember how resilient you are—you’ve navigate...,1,control_ai_draft
4,b417dba7-2b67-4496-b985-fc67c15aa748,"It’s okay to slow down, take a breath, and giv...",1,control_ai_draft


### SPlitting thse sentences into training and validation

In [32]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        sentence_df,
        y=sentence_df["label"],
        groups=sentence_df["participant_id"]
    )
)

train_df = sentence_df.iloc[train_idx].reset_index(drop=True)
val_df = sentence_df.iloc[val_idx].reset_index(drop=True)

train_participants = set(train_df["participant_id"])
val_participants = set(val_df["participant_id"])

print("Training sentences:", len(train_df))
print("Validation sentences:", len(val_df))
print("Training participants:", len(train_participants))
print("Validation participants:", len(val_participants))
print("Participants shared:", len(train_participants & val_participants))
print("\nTraining labels:")
print(train_df["label"].value_counts())
print("\nValidation labels:")
print(val_df["label"].value_counts())

Training sentences: 3528
Validation sentences: 913
Training participants: 45
Validation participants: 12
Participants shared: 0

Training labels:
label
1    2650
0     878
Name: count, dtype: int64

Validation labels:
label
1    689
0    224
Name: count, dtype: int64


In [35]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "HUMAN", 1: "AI"},
    label2id={"HUMAN": 0, "AI": 1}
)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

train_hf = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
).rename_column("label", "labels")

val_hf = Dataset.from_pandas(
    val_df[["text", "label"]],
    preserve_index=False
).rename_column("label", "labels")

train_hf = train_hf.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

val_hf = val_hf.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Training rows:", len(train_hf))
print("Validation rows:", len(val_hf))
print("Model:", model_name)
print(train_hf[0].keys())

c:\Users\Samarth PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Samarth PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Samarth PC\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as 

Training rows: 3528
Validation rows: 913
Model: roberta-base
dict_keys(['labels', 'input_ids', 'attention_mask'])


### g

In [36]:
import numpy as np
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support
)
from transformers import Trainer, TrainingArguments

# Give the less frequent human class more influence during training.
label_counts = train_df["label"].value_counts().sort_index()
class_weights = torch.tensor(
    [
        len(train_df) / (2 * label_counts[0]),
        len(train_df) / (2 * label_counts[1])
    ],
    dtype=torch.float
)

def weighted_loss(outputs, labels, num_items_in_batch):
    loss_sum = F.cross_entropy(
        outputs.logits,
        labels,
        weight=class_weights.to(outputs.logits.device),
        reduction="sum"
    )
    return loss_sum / num_items_in_batch

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        labels=[0, 1],
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(
            labels, predictions, average="macro", zero_division=0
        ),
        "human_f1": f1[0],
        "ai_f1": f1[1],
        "human_precision": precision[0],
        "human_recall": recall[0],
        "ai_precision": precision[1],
        "ai_recall": recall[1]
    }

training_args = TrainingArguments(
    output_dir="roberta_ai_human_checkpoints",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    logging_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    seed=42,
    dataloader_num_workers=0
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    compute_loss_func=weighted_loss
)

print("Class weights [human, AI]:", class_weights.tolist())
print("CUDA available:", torch.cuda.is_available())

Class weights [human, AI]: [2.0091116428375244, 0.6656603813171387]
CUDA available: False


In [37]:
training_args.num_train_epochs = 1

train_result = trainer.train()
eval_result = trainer.evaluate()

print(eval_result)

c:\Users\Samarth PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Human F1,Ai F1,Human Precision,Human Recall,Ai Precision,Ai Recall
1,0.510942,0.354002,0.920044,0.883779,0.818859,0.948700,0.921788,0.736607,0.919619,0.979681


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.04it/s]
c:\Users\Samarth PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,Human F1,Ai F1,Human Precision,Human Recall,Ai Precision,Ai Recall
0.510942,0.354002,1,0.920044,0.883779,0.818859,0.948700,0.921788,0.736607,0.919619,0.979681


{'eval_loss': 0.3540017604827881, 'eval_accuracy': 0.9200438116100766, 'eval_f1_macro': 0.8837792452599879, 'eval_human_f1': 0.8188585607940446, 'eval_ai_f1': 0.9486999297259311, 'eval_human_precision': 0.9217877094972067, 'eval_human_recall': 0.7366071428571429, 'eval_ai_precision': 0.9196185286103542, 'eval_ai_recall': 0.9796806966618288}


In [38]:
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix

pred_output = trainer.predict(val_hf)
logits = pred_output.predictions
predicted_labels = np.argmax(logits, axis=1)
prob_ai = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

print("Confusion matrix (rows = true, columns = predicted; [human, AI]):")
print(confusion_matrix(val_df["label"], predicted_labels, labels=[0, 1]))

print("\nPer-class report:")
print(classification_report(
    val_df["label"],
    predicted_labels,
    labels=[0, 1],
    target_names=["Human", "AI"],
    zero_division=0
))

val_results = val_df.copy()
val_results["predicted_label"] = predicted_labels
val_results["probability_ai"] = prob_ai

print("\nMisclassified validation sentences:")
display(
    val_results[val_results["label"] != val_results["predicted_label"]][
        ["source", "label", "predicted_label", "probability_ai", "text"]
    ].head(10)
)

c:\Users\Samarth PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Confusion matrix (rows = true, columns = predicted; [human, AI]):
[[165  59]
 [ 14 675]]

Per-class report:
              precision    recall  f1-score   support

       Human       0.92      0.74      0.82       224
          AI       0.92      0.98      0.95       689

    accuracy                           0.92       913
   macro avg       0.92      0.86      0.88       913
weighted avg       0.92      0.92      0.92       913


Misclassified validation sentences:


,source,label,predicted_label,probability_ai,text
3,control_ai_draft,1,0,0.099526,I care about you and never meant for my action...
57,pure_final,0,1,0.933432,Please continue to take care of yourself and g...
60,pure_final,0,1,0.988639,I can't imagine how difficult this must be for...
63,pure_final,0,1,0.984078,Even though words can't ease the sorrow entire...
64,pure_final,0,1,0.960308,Her memory will continue to live in your heart...
65,pure_final,0,1,0.617723,Take care of yourself and allow yourself to gr...
122,pure_final,0,1,0.681574,Hello Paul!
123,pure_final,0,1,0.518300,Hope you are doing great.
125,pure_final,0,1,0.962249,I can't imagine the financial support you offe...
129,pure_final,0,1,0.853279,Thank you so much Paul for standing with me du...


In [39]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Separate false alarms and missed AI sentences
false_positives = val_results[
    (val_results["label"] == 0) &
    (val_results["predicted_label"] == 1)
]

false_negatives = val_results[
    (val_results["label"] == 1) &
    (val_results["predicted_label"] == 0)
]

print("Human sentences incorrectly flagged as AI:", len(false_positives))
print("AI sentences incorrectly classified as human:", len(false_negatives))

print("\nFalse alarms by source:")
print(false_positives["source"].value_counts())

print("\nFalse alarms by participant (largest counts first):")
print(false_positives["participant_id"].value_counts().head(10))

print("\nMost confident false alarms:")
display(
    false_positives.sort_values("probability_ai", ascending=False)[
        ["participant_id", "source", "probability_ai", "text"]
    ].head(15)
)

print("\nAI sentences most confidently missed:")
display(
    false_negatives.sort_values("probability_ai")[
        ["participant_id", "source", "probability_ai", "text"]
    ].head(15)
)

# Compare possible cutoffs for calling a sentence AI
threshold_rows = []

for threshold in [0.35, 0.50, 0.60, 0.70, 0.80]:
    threshold_predictions = (
        val_results["probability_ai"] >= threshold
    ).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        val_results["label"],
        threshold_predictions,
        labels=[0, 1],
        zero_division=0
    )

    threshold_rows.append({
        "AI_cutoff": threshold,
        "accuracy": accuracy_score(
            val_results["label"], threshold_predictions
        ),
        "macro_F1": (f1[0] + f1[1]) / 2,
        "human_recall": recall[0],
        "AI_recall": recall[1],
        "human_precision": precision[0],
        "AI_precision": precision[1]
    })

threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(3))

Human sentences incorrectly flagged as AI: 59
AI sentences incorrectly classified as human: 14

False alarms by source:
source
pure_final    59
Name: count, dtype: int64

False alarms by participant (largest counts first):
participant_id
36f7c04d-9eb6-41fa-ae60-a6ad6fa1c429    11
f5d09877-f461-4169-a291-79c17218071d    10
379876ef-ea66-4914-90c7-052d13ad9036    10
8ba8d7da-bbcf-44c8-8887-d6b083446deb     7
c9afe4ed-b46e-4919-a89b-9c3639640775     5
95648190-a022-43f7-a299-7d7c22e55289     4
0e4ed97b-533a-4c35-9268-2610361186c3     3
9d164220-c497-48ff-8497-55c050213e55     3
9473c36d-e078-4a5b-a106-a4de719f1c1b     3
03acc614-1708-42a6-b142-9ef7a0f0ce4e     2
Name: count, dtype: int64

Most confident false alarms:


,participant_id,source,probability_ai,text
296,379876ef-ea66-4914-90c7-052d13ad9036,pure_final,0.991672,"As we take this plunge, may the love we have a..."
60,c9afe4ed-b46e-4919-a89b-9c3639640775,pure_final,0.988639,I can't imagine how difficult this must be for...
63,c9afe4ed-b46e-4919-a89b-9c3639640775,pure_final,0.984078,Even though words can't ease the sorrow entire...
215,36f7c04d-9eb6-41fa-ae60-a6ad6fa1c429,pure_final,0.976500,Thank you for bringing joy and happiness into ...
509,8ba8d7da-bbcf-44c8-8887-d6b083446deb,pure_final,0.975101,Your passion for photography still amazes me.
745,95648190-a022-43f7-a299-7d7c22e55289,pure_final,0.973177,I am looking forward to our next adventure and...
676,9473c36d-e078-4a5b-a106-a4de719f1c1b,pure_final,0.967027,your insight and perspective have stuck with me .
297,379876ef-ea66-4914-90c7-052d13ad9036,pure_final,0.967006,Dear Nina\n\nI am reaching out with this lette...
132,f5d09877-f461-4169-a291-79c17218071d,pure_final,0.966849,I really felt proud by your presence during th...
125,f5d09877-f461-4169-a291-79c17218071d,pure_final,0.962249,I can't imagine the financial support you offe...



AI sentences most confidently missed:


,participant_id,source,probability_ai,text
190,36f7c04d-9eb6-41fa-ae60-a6ad6fa1c429,pure_ai_draft,0.012805,"Carol, you are one of the kindest and most car..."
339,16e11b3b-7143-40ed-ba2c-c94e96d5be86,control_ai_draft,0.038492,It feels we picked up where we left off.
709,95648190-a022-43f7-a299-7d7c22e55289,control_ai_draft,0.044707,"Jim, from the moment our friends introduced us..."
240,379876ef-ea66-4914-90c7-052d13ad9036,control_ai_draft,0.046360,I am so sorry for your loss.
841,9d164220-c497-48ff-8497-55c050213e55,control_ai_draft,0.080757,Things will get better soon.
3,c9afe4ed-b46e-4919-a89b-9c3639640775,control_ai_draft,0.099526,I care about you and never meant for my action...
185,36f7c04d-9eb6-41fa-ae60-a6ad6fa1c429,control_ai_draft,0.136395,I am holding you close in thought.
421,d1df814b-04d7-45ef-afa9-19333672b460,pure_ai_draft,0.161096,"Still, that does not excuse what I did."
188,36f7c04d-9eb6-41fa-ae60-a6ad6fa1c429,pure_ai_draft,0.258198,I still remember her kneeling beside me when I...
837,9d164220-c497-48ff-8497-55c050213e55,control_ai_draft,0.278462,I believe in you completely.


,AI_cutoff,accuracy,macro_F1,human_recall,AI_recall,human_precision,AI_precision
0,0.35,0.919,0.880,0.714,0.985,0.941,0.914
1,0.50,0.920,0.884,0.737,0.980,0.922,0.920
2,0.60,0.921,0.887,0.754,0.975,0.909,0.924
3,0.70,0.924,0.893,0.781,0.971,0.897,0.932
4,0.80,0.926,0.897,0.812,0.962,0.875,0.940


## Testing Participants

In [40]:
test_records = {}

for file in test_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    participant_id = data["user_info"]["id"]
    test_records[participant_id] = {"control": [], "pure": []}

    for response in data["responses"].values():
        if not response.get("scenario"):
            continue

        if response["model_generation_shown"] == 0:
            test_records[participant_id]["pure"].append(
                (response["model_generation"], response["final_version"])
            )
        elif response["model_generation_shown"] == 1:
            test_records[participant_id]["control"].append(
                (response["model_generation"], response["final_version"])
            )

test_sentence_examples = []

for participant_id, participant_data in test_records.items():

    # AI-written drafts receive label 1.
    for condition in ("control", "pure"):
        for ai_draft, final_text in participant_data[condition]:
            if not isinstance(ai_draft, str):
                continue

            for sentence in sent_tokenize(ai_draft):
                sentence = sentence.strip()
                if sentence:
                    test_sentence_examples.append({
                        "participant_id": participant_id,
                        "text": sentence,
                        "label": 1,
                        "source": f"{condition}_ai_draft"
                    })

    # Pure final texts receive label 0.
    for ai_draft, human_text in participant_data["pure"]:
        if not isinstance(human_text, str):
            continue

        for sentence in sent_tokenize(human_text):
            sentence = sentence.strip()
            if sentence:
                test_sentence_examples.append({
                    "participant_id": participant_id,
                    "text": sentence,
                    "label": 0,
                    "source": "pure_final"
                })

test_sentence_df = pd.DataFrame(test_sentence_examples)

print("Held-out participants:", len(test_records))
print("Held-out sentences:", len(test_sentence_df))
print(test_sentence_df["label"].value_counts())
print(
    "Participants shared with training/validation:",
    len(set(test_sentence_df["participant_id"]) &
        (train_participants | val_participants))
)

Held-out participants: 24
Held-out sentences: 1895
label
1    1407
0     488
Name: count, dtype: int64
Participants shared with training/validation: 0


In [41]:
from sklearn.metrics import classification_report, confusion_matrix

test_hf = Dataset.from_pandas(
    test_sentence_df[["text", "label"]],
    preserve_index=False
).rename_column("label", "labels")

test_hf = test_hf.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

test_output = trainer.predict(test_hf)
test_logits = test_output.predictions

test_prob_ai = torch.softmax(
    torch.tensor(test_logits), dim=1
)[:, 1].numpy()

chosen_cutoff = 0.80
test_predictions = (test_prob_ai >= chosen_cutoff).astype(int)

print("AI cutoff:", chosen_cutoff)
print("Confusion matrix (rows=true, columns=predicted; [human, AI]):")
print(confusion_matrix(
    test_sentence_df["label"],
    test_predictions,
    labels=[0, 1]
))

print("\nHeld-out classification report:")
print(classification_report(
    test_sentence_df["label"],
    test_predictions,
    labels=[0, 1],
    target_names=["Human", "AI"],
    zero_division=0
))

Map: 100%|██████████| 1895/1895 [00:00<00:00, 64205.62 examples/s]
c:\Users\Samarth PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


AI cutoff: 0.8
Confusion matrix (rows=true, columns=predicted; [human, AI]):
[[ 380  108]
 [  78 1329]]

Held-out classification report:
              precision    recall  f1-score   support

       Human       0.83      0.78      0.80       488
          AI       0.92      0.94      0.93      1407

    accuracy                           0.90      1895
   macro avg       0.88      0.86      0.87      1895
weighted avg       0.90      0.90      0.90      1895



In [42]:
model_dir = os.path.join(path, "roberta_ai_human_model")
os.makedirs(model_dir, exist_ok=True)

# trainer has the best validation checkpoint loaded
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)

test_report = classification_report(
    test_sentence_df["label"],
    test_predictions,
    labels=[0, 1],
    target_names=["Human", "AI"],
    output_dict=True,
    zero_division=0
)

with open(os.path.join(model_dir, "evaluation.json"), "w", encoding="utf-8") as f:
    json.dump({
        "ai_probability_cutoff": 0.80,
        "cutoff_selected_using": "validation participants",
        "test_participants": len(test_records),
        "test_sentences": len(test_sentence_df),
        "metrics": test_report
    }, f, indent=2)

print("Saved model and evaluation files to:", model_dir)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.09it/s]


Saved model and evaluation files to: D:\Academics Semester 5\DS501 - Machine Learning\Project\PersonalizationOfLLMs\roberta_ai_human_model
